# Cortex model access audit and allowlist-to-RBAC migration

**Goal:** find everything that depends on the old `CORTEX_MODELS_ALLOWLIST` account parameter, and prepare (then optionally apply) the model RBAC grants that replace it.

**Background (from Snowflake's release notes):** access control based on `CORTEX_MODELS_ALLOWLIST` is being removed. Model RBAC (application roles in the `SNOWFLAKE` application, such as `SNOWFLAKE."CORTEX-MODEL-ROLE-ALL"`) becomes the only mechanism, and it now also covers embedding models. An allowlist of `All` is equivalent to granting `CORTEX-MODEL-ROLE-ALL` to `PUBLIC`; a list of models is equivalent to granting those model application roles to `PUBLIC`. Always test with a role other than ACCOUNTADMIN, which always has access to every model.

> I could not open the "Migrate from the allowlist to RBAC" section directly when preparing this. Compare the generated steps against that section before running anything with `DRY_RUN = False`.

## How this notebook is organised
| Part | What it does | Changes anything? |
|---|---|---|
| 0 | Config and connection | No |
| 1-2 | Allowlist value, baseline Cortex privileges, cross-region setting | No |
| 3 | Model application roles and who holds them | No |
| 4 | Real usage: which roles call which AI functions and models | No |
| 5 | Embeddings, Cortex Search, stored procedures, recent failures | No |
| 6 | Builds the proposed GRANT script | No (dry run) |
| 7 | Applies grants, validates with a non-ACCOUNTADMIN role, sets allowlist to `None` | **Only when you flip the flags** |

## Access you will likely need
Each cell reports `[NO ACCESS / ERROR]` instead of crashing, so you can run the notebook with partial access and see what is blocked.

| Needed for | Typical requirement |
|---|---|
| Parts 1-3 (SHOW commands) | A role that can see account parameters and the `SNOWFLAKE` application roles and their grants |
| Parts 4-5 (usage, queries, procedures) | Access to `SNOWFLAKE.ACCOUNT_USAGE` (imported privileges on the `SNOWFLAKE` database, or the database roles that cover those views, e.g. USAGE_VIEWER / GOVERNANCE_VIEWER) |
| Part 7 grants | SECURITYADMIN or ACCOUNTADMIN (needs the ability to grant application roles) |
| Part 7 `ALTER ACCOUNT` | ACCOUNTADMIN |

## PHI note
This is a hospital environment. The notebook never stores or exports query text: query text is only used inside SQL to extract function names, and exports contain aggregates only.

## 0. Configuration and connection

In [ ]:
# --- Configuration ---------------------------------------------------------
DRY_RUN = True                         # True = only print statements. Set False ONLY when approved.
LOOKBACK_DAYS = 90                     # history window (ACCOUNT_USAGE keeps 365 days; has ~up to 3h latency)
PRESERVE_BEHAVIOR_VIA_PUBLIC = False   # True = also generate PUBLIC grants that mirror the old allowlist
                                       # (fast and safe for continuity, but broad; prefer targeted role grants)
TEST_ROLE = None                       # a NON-ACCOUNTADMIN role to validate with, e.g. "ANALYST_ROLE"
TEST_MODEL = "llama3.1-8b"             # small model used for the validation call; change to one you use
TEST_EMBED_MODEL = "snowflake-arctic-embed-m-v1.5"
CONFIRM_SET_ALLOWLIST_NONE = False     # final cutover flag (see Part 7)
OUTPUT_DIR = "audit_output"            # CSV exports (aggregates only, no query text)

In [ ]:
import os, re, pathlib
import pandas as pd
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

def _fetch(cur):
    cols = [c[0] for c in cur.description] if cur.description else []
    return pd.DataFrame(cur.fetchall(), columns=cols)

try:
    # Inside a Snowflake Notebook
    from snowflake.snowpark.context import get_active_session
    _session = get_active_session()
    def run(sql):
        return _session.sql(sql).to_pandas()
    ENV = "snowflake-notebook"
except Exception:
    # Local Jupyter: set SNOWFLAKE_ACCOUNT, SNOWFLAKE_USER (and optionally SNOWFLAKE_ROLE / SNOWFLAKE_WAREHOUSE)
    import snowflake.connector
    _conn = snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"],
        user=os.environ["SNOWFLAKE_USER"],
        authenticator=os.environ.get("SNOWFLAKE_AUTHENTICATOR", "externalbrowser"),  # SSO
        role=os.environ.get("SNOWFLAKE_ROLE"),
        warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE"),
    )
    def run(sql):
        cur = _conn.cursor()
        try:
            cur.execute(sql)
            return _fetch(cur)
        finally:
            cur.close()
    ENV = "local"

FINDINGS = []   # (area, severity, message)

def note(area, severity, msg):
    FINDINGS.append((area, severity, msg))
    print(f"[{severity.upper()}] {area}: {msg}")

def safe(label, sql):
    """Run a query; on a privilege/other error, record it and return None instead of stopping."""
    try:
        df = run(sql)
        df.columns = [str(c).lower() for c in df.columns]
        return df
    except Exception as e:
        msg = str(e).strip().splitlines()[0][:220]
        note(label, "blocked", msg)
        return None

print("Environment:", ENV)
ctx = safe("session context", "SELECT CURRENT_ACCOUNT() AS account, CURRENT_REGION() AS region, CURRENT_ROLE() AS role, CURRENT_USER() AS usr")
ctx

## 1. Current allowlist value and cross-region setting

In [ ]:
ALLOWLIST_VALUE = None

df = safe("allowlist parameter", "SHOW PARAMETERS LIKE 'CORTEX_MODELS_ALLOWLIST' IN ACCOUNT")
if df is not None:
    display(df)
    if df.empty:
        note("allowlist", "info", "Parameter not returned. It may already be removed from this account.")
    else:
        ALLOWLIST_VALUE = str(df.iloc[0]["value"])
        if ALLOWLIST_VALUE.strip().lower() == "none":
            note("allowlist", "ok", "Already 'None': account uses model RBAC exclusively. Remaining work is verifying role grants (Parts 3-5).")
        elif ALLOWLIST_VALUE.strip().lower() == "all":
            note("allowlist", "high", "'All' -> equivalent to CORTEX-MODEL-ROLE-ALL granted to PUBLIC. Everyone currently relies on this; migration needed.")
        else:
            note("allowlist", "high", f"Allowlist = {ALLOWLIST_VALUE!r} -> equivalent to those model roles granted to PUBLIC. Migration needed.")

xr = safe("cross-region parameter", "SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT")
if xr is not None:
    display(xr[["key", "value", "level"]] if not xr.empty else xr)
    if not xr.empty and str(xr.iloc[0]["value"]).upper() not in ("DISABLED", ""):
        note("cross-region", "review", f"Cross-region inference is {xr.iloc[0]['value']!r}. Confirm this is approved by security/compliance for a hospital workload.")

## 2. Baseline Cortex privileges (who can call AI functions at all)

In [ ]:
baseline = {}
for dbrole in ["CORTEX_USER", "CORTEX_EMBED_USER"]:
    df = safe(f"grants of SNOWFLAKE.{dbrole}", f"SHOW GRANTS OF DATABASE ROLE SNOWFLAKE.{dbrole}")
    baseline[dbrole] = df
    if df is not None:
        print(f"\n--- SNOWFLAKE.{dbrole} granted to ---")
        display(df[["granted_to", "grantee_name"]] if not df.empty else df)
        if not df.empty and (df["grantee_name"].astype(str).str.upper() == "PUBLIC").any():
            note(f"{dbrole}", "review", "Granted to PUBLIC: every user can call Cortex functions. Decide whether to restrict to specific roles.")

## 3. Model application roles and who holds them

In [ ]:
MODEL_ROLE_PREFIX = "CORTEX-MODEL-ROLE-"
model_app_roles = []
grant_rows = []

df = safe("application roles in SNOWFLAKE", "SHOW APPLICATION ROLES IN APPLICATION SNOWFLAKE")
if df is not None:
    model_app_roles = sorted(n for n in df["name"] if str(n).upper().startswith(MODEL_ROLE_PREFIX))
    print(f"{len(model_app_roles)} model application roles found")
    for r in model_app_roles:
        g = safe(f"grants of {r}", f'SHOW GRANTS OF APPLICATION ROLE SNOWFLAKE."{r}"')
        if g is not None:
            for _, row in g.iterrows():
                grant_rows.append((r, row.get("granted_to"), row.get("grantee_name")))

model_grants = pd.DataFrame(grant_rows, columns=["app_role", "granted_to", "grantee"])
if not model_grants.empty:
    display(model_grants.groupby(["grantee", "granted_to"])["app_role"].apply(lambda s: ", ".join(sorted(s))).reset_index())
    pub = model_grants[model_grants["grantee"].astype(str).str.upper() == "PUBLIC"]
    if not pub.empty:
        note("model RBAC", "review", f"{len(pub)} model role(s) already granted to PUBLIC: {', '.join(pub['app_role'])}")
else:
    print("No model-role grants visible (either none exist or access is blocked).")

## 4. Real usage: who calls which AI functions and models

Uses `ACCOUNT_USAGE.QUERY_HISTORY` and `CORTEX_FUNCTIONS_QUERY_USAGE_HISTORY`. View and column names can differ by account or release; any error is reported and the notebook continues.

In [ ]:
AI_FN_REGEX = r"AI_(COMPLETE|CLASSIFY|FILTER|AGG|SUMMARIZE_AGG|SUMMARIZE|EXTRACT|TRANSLATE|SENTIMENT|TRANSCRIBE|PARSE_DOCUMENT|REDACT|EMBED|SIMILARITY)|SNOWFLAKE\.CORTEX\.[A-Z_0-9]+"
AI_DETECT = rf"$$.*({AI_FN_REGEX}|CORTEX).*$$"
AI_EXTRACT = rf"$${AI_FN_REGEX}$$"

fn_usage = safe("function usage by role", rf"""
WITH q AS (
  SELECT query_id, user_name, role_name, start_time, execution_status, query_text
  FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
  WHERE start_time >= DATEADD(day, -{LOOKBACK_DAYS}, CURRENT_TIMESTAMP())
    AND REGEXP_LIKE(query_text, {AI_DETECT}, 'is')
), f AS (
  SELECT DISTINCT q.query_id, q.user_name, q.role_name, q.start_time, q.execution_status,
         UPPER(x.value::string) AS function_name
  FROM q, LATERAL FLATTEN(input => REGEXP_SUBSTR_ALL(q.query_text, {AI_EXTRACT}, 1, 1, 'i')) x
)
SELECT role_name, function_name, execution_status,
       COUNT(*) AS calls, COUNT(DISTINCT user_name) AS users,
       MIN(start_time) AS first_seen, MAX(start_time) AS last_seen
FROM f
GROUP BY 1, 2, 3
ORDER BY calls DESC
""")
if fn_usage is not None:
    display(fn_usage)
    note("usage", "info", f"{fn_usage['role_name'].nunique()} role(s) used AI functions in the last {LOOKBACK_DAYS} days.")

In [ ]:
model_usage = safe("model usage by role", rf"""
SELECT q.role_name, u.function_name, u.model_name,
       COUNT(DISTINCT u.query_id) AS queries,
       SUM(u.tokens) AS tokens, SUM(u.token_credits) AS credits
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_FUNCTIONS_QUERY_USAGE_HISTORY u
JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY q ON q.query_id = u.query_id
WHERE q.start_time >= DATEADD(day, -{LOOKBACK_DAYS}, CURRENT_TIMESTAMP())
GROUP BY 1, 2, 3
ORDER BY queries DESC
""")
if model_usage is not None:
    display(model_usage)

## 5. Embeddings, Cortex Search, stored procedures, recent failures

In [ ]:
# Embedding usage (embedding models had NO model RBAC before; they are the most likely silent breakage)
if fn_usage is not None:
    emb = fn_usage[fn_usage["function_name"].str.contains("EMBED", na=False)]
    print("Embedding function usage:")
    display(emb if not emb.empty else pd.DataFrame({"result": ["none found in window"]}))
    if not emb.empty:
        note("embeddings", "high", f"{emb['role_name'].nunique()} role(s) call embedding functions and will need embedding model grants.")

# Cortex Search services (managed embeddings under the hood)
css = safe("cortex search services", "SHOW CORTEX SEARCH SERVICES IN ACCOUNT")
if css is not None:
    display(css[[c for c in ["name", "database_name", "schema_name", "owner", "created_on"] if c in css.columns]] if not css.empty else css)
    if not css.empty:
        note("cortex search", "review", f"{len(css)} Cortex Search service(s) exist. Confirm each service owner/query role has embedding model access.")

In [ ]:
# Stored procedures that call AI functions (EXECUTE AS RESTRICTED CALLER needs extra privileges on the creating role)
procs = safe("procedures using AI functions", rf"""
SELECT procedure_catalog, procedure_schema, procedure_name, argument_signature,
       procedure_owner, procedure_language, created, last_altered
FROM SNOWFLAKE.ACCOUNT_USAGE.PROCEDURES
WHERE deleted IS NULL
  AND REGEXP_LIKE(procedure_definition, {AI_DETECT}, 'is')
ORDER BY 1, 2, 3
""")
if procs is not None:
    display(procs)
    restricted = []
    for _, p in procs.head(100).iterrows():
        types = ", ".join(a.strip().split(" ", 1)[-1] for a in re.sub(r"^\(|\)$", "", str(p["argument_signature"])).split(",") if a.strip())
        fq = f'{p["procedure_catalog"]}.{p["procedure_schema"]}.{p["procedure_name"]}({types})'
        ddl = safe(f"ddl {fq}", f"SELECT GET_DDL('PROCEDURE', '{fq}') AS ddl")
        if ddl is not None and not ddl.empty and re.search(r"EXECUTE\s+AS\s+RESTRICTED\s+CALLER", str(ddl.iloc[0, 0]), re.I):
            restricted.append(fq)
    if restricted:
        note("procedures", "high", f"{len(restricted)} procedure(s) use EXECUTE AS RESTRICTED CALLER and call AI functions: {restricted}. Check the creating role's privileges against the Snowflake doc.")
    else:
        note("procedures", "info", "No RESTRICTED CALLER procedures detected among those checked (or DDL was not readable).")

In [ ]:
failures = safe("recent AI-related failures", rf"""
SELECT role_name, user_name, error_code, LEFT(error_message, 200) AS error_message,
       COUNT(*) AS failures, MAX(start_time) AS last_failure
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time >= DATEADD(day, -30, CURRENT_TIMESTAMP())
  AND execution_status = 'FAIL'
  AND REGEXP_LIKE(query_text, {AI_DETECT}, 'is')
  AND REGEXP_LIKE(error_message, $$.*(model|cortex|privilege|not authorized|insufficient|access).*$$, 'is')
GROUP BY 1, 2, 3, 4
ORDER BY failures DESC
""")
if failures is not None:
    display(failures)
    if not failures.empty:
        note("failures", "high", f"{int(failures['failures'].sum())} AI-related access-style failures in the last 30 days. Likely RBAC gaps already biting.")

## 6. Build the proposed grant script (no changes yet)

Matching model names to application role names is done by normalising both names. **Exact** matches are generated automatically; **fuzzy** matches are listed for you to confirm; anything unmatched needs manual review. Grants inherited through role hierarchies are not resolved here, so a role may look uncovered when a parent role already has the grant. Check with `SHOW GRANTS TO ROLE` before applying.

In [ ]:
def norm(s):
    return re.sub(r"[^A-Z0-9]", "", str(s).upper())

def match_app_role(model):
    m = norm(model)
    if not m:
        return None, "no model name"
    cands = [(r, norm(r[len(MODEL_ROLE_PREFIX):])) for r in model_app_roles]
    for r, k in cands:
        if k == m:
            return r, "exact"
    for r, k in cands:
        if k != "ALL" and (m in k or k in m):
            return r, "fuzzy"
    return None, "unmatched"

def is_covered(role, app_role):
    if model_grants.empty:
        return False
    g = model_grants[model_grants["grantee"].astype(str).isin([str(role), "PUBLIC"]) &
                     model_grants["app_role"].isin([app_role, MODEL_ROLE_PREFIX + "ALL"])]
    return not g.empty

proposal_rows = []
if model_usage is not None and not model_usage.empty:
    for (role, model), _ in model_usage.groupby(["role_name", "model_name"], dropna=False):
        if role is None or str(role).upper() == "ACCOUNTADMIN":
            continue
        app_role, how = match_app_role(model)
        covered = bool(app_role) and is_covered(role, app_role)
        proposal_rows.append((role, model, app_role, how, covered))

proposal = pd.DataFrame(proposal_rows, columns=["role", "model", "app_role", "match", "already_covered"])
display(proposal)

need_exact = proposal[(proposal["match"] == "exact") & (~proposal["already_covered"])] if not proposal.empty else proposal
need_review = proposal[(proposal["match"] != "exact") & (~proposal["already_covered"])] if not proposal.empty else proposal

statements, rollback = [], []
for _, r in need_exact.iterrows():
    statements.append(f'GRANT APPLICATION ROLE SNOWFLAKE."{r["app_role"]}" TO ROLE "{r["role"]}";')
    rollback.append(f'REVOKE APPLICATION ROLE SNOWFLAKE."{r["app_role"]}" FROM ROLE "{r["role"]}";')

# Optional: mirror the old allowlist via PUBLIC (broad)
if PRESERVE_BEHAVIOR_VIA_PUBLIC and ALLOWLIST_VALUE and ALLOWLIST_VALUE.strip().lower() != "none":
    if ALLOWLIST_VALUE.strip().lower() == "all":
        targets = [MODEL_ROLE_PREFIX + "ALL"]
    else:
        targets = [a for a, _ in (match_app_role(m.strip()) for m in ALLOWLIST_VALUE.split(",")) if a]
    for a in targets:
        statements.append(f'GRANT APPLICATION ROLE SNOWFLAKE."{a}" TO ROLE PUBLIC;')
        rollback.append(f'REVOKE APPLICATION ROLE SNOWFLAKE."{a}" FROM ROLE PUBLIC;')

statements = list(dict.fromkeys(statements)); rollback = list(dict.fromkeys(rollback))
print("\n-- PROPOSED GRANTS --"); print("\n".join(statements) or "(none generated)")
print("\n-- ROLLBACK --"); print("\n".join(rollback) or "(none)")
print(f"\n{len(need_review)} usage row(s) need manual review (fuzzy/unmatched/no model name):")
display(need_review)

## 7. Apply, validate, cut over (gated)

Order matters: **grant -> validate with a non-ACCOUNTADMIN role -> set allowlist to `None`**. Nothing below changes anything while `DRY_RUN = True`.

In [ ]:
def apply(stmts):
    for s in stmts:
        if DRY_RUN:
            print("[DRY RUN]", s)
        else:
            run(s.rstrip(";"))
            print("[APPLIED]", s)

apply(statements)

In [ ]:
# Validate as a non-ACCOUNTADMIN role (ACCOUNTADMIN always has access to all models, so it proves nothing)
def validate(role):
    if not role:
        print("Set TEST_ROLE in the config cell first."); return
    if role.upper() == "ACCOUNTADMIN":
        print("Use a non-ACCOUNTADMIN role."); return
    original = run("SELECT CURRENT_ROLE() AS r").iloc[0, 0]
    try:
        run(f'USE ROLE "{role}"')
        tests = {
            f"AI_COMPLETE({TEST_MODEL})": f"SELECT AI_COMPLETE('{TEST_MODEL}', 'Reply with the single word OK') AS out",
            f"AI_EMBED({TEST_EMBED_MODEL})": f"SELECT ARRAY_SIZE(AI_EMBED('{TEST_EMBED_MODEL}', 'test')::ARRAY) AS dims",
        }
        for label, sql in tests.items():
            try:
                print("PASS", label, "->", str(run(sql).iloc[0, 0])[:60])
            except Exception as e:
                print("FAIL", label, "->", str(e).splitlines()[0][:200])
    finally:
        run(f'USE ROLE "{original}"')

validate(TEST_ROLE)

In [ ]:
# Final cutover: set the allowlist to 'None' (requires ACCOUNTADMIN). Only after validation passes.
# Note: Snowflake documents the allowlist as being removed; if the parameter is already inert, this is a no-op.
if CONFIRM_SET_ALLOWLIST_NONE and not DRY_RUN:
    print("Previous value (save for rollback):", ALLOWLIST_VALUE)
    run("ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'None'")
    print("[APPLIED] ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'None'")
else:
    print("[SKIPPED] Set DRY_RUN=False and CONFIRM_SET_ALLOWLIST_NONE=True after validation to run the cutover.")
    print(f"Would run: ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'None';   (current: {ALLOWLIST_VALUE!r})")

## 8. Summary and export (aggregates only)

In [ ]:
summary = pd.DataFrame(FINDINGS, columns=["area", "severity", "message"])
order = {"high": 0, "blocked": 1, "review": 2, "info": 3, "ok": 4}
summary = summary.sort_values("severity", key=lambda s: s.map(order)).reset_index(drop=True)
display(summary)

out = pathlib.Path(OUTPUT_DIR); out.mkdir(exist_ok=True)
summary.to_csv(out / "findings.csv", index=False)
for name, d in {"function_usage": fn_usage, "model_usage": model_usage, "proposal": proposal, "failures": failures}.items():
    if d is not None and not d.empty:
        d.to_csv(out / f"{name}.csv", index=False)
(out / "proposed_grants.sql").write_text("\n".join(statements) + "\n")
(out / "rollback.sql").write_text("\n".join(rollback) + "\n")
print("Exported to", out.resolve())
print("Items marked 'blocked' are what you still need access for.")